In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import LabelEncoder

pd.set_option("display.max_columns", None)

RANDOM_STATE = 42

In [4]:
# Loading the cleaned dataset

df = pd.read_csv("../data/cleaned/deforestation_cleaned.csv")

print("Dataset Shape:", df.shape)


Dataset Shape: (1397, 12)


In [5]:
# Converting numeric columns stored as strings

numeric_columns = [
    "Annual_Rainfall_mm",
    "Population_Density"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Check for missing values created during conversion
print("\nMissing Values")
print(df[numeric_columns].isnull().sum())

df.head()


Missing Values
Annual_Rainfall_mm    1
Population_Density    1
dtype: int64


,Record_ID,Observation_Date,Region,District,Forest_Type,Forest_Area_ha,Tree_Cover_Loss_ha,Annual_Rainfall_mm,Population_Density,Fire_Incidents,Illegal_Logging,Deforestation_Risk
0,REC00001,2024-06-27,Central,Wakiso,Mangrove,4179.46,235.27,1358.1,657.0,16,Yes,High
1,REC00002,2021-04-30,Eastern,Mbale,Mangrove,2072.03,14.36,1605.3,266.0,25,Yes,High
2,REC00003,2022-01-20,Central,Mukono,Woodland,7417.11,543.23,1471.4,245.0,16,Yes,High
3,REC00004,2021-07-17,Western,Mbarara,Montane Forest,5260.01,281.86,1609.9,434.0,11,Yes,High
4,REC00005,2023-07-07,Eastern,Mbale,Dry Forest,9793.89,575.77,1353.5,647.0,6,No,High


The following features were selected because they describe environmental conditions and human activities associated with forest degradation.

In [6]:
baseline_features = [
    "Forest_Area_ha",
    "Tree_Cover_Loss_ha",
    "Annual_Rainfall_mm",
    "Population_Density",
    "Fire_Incidents"
]

print("Baseline Features")

baseline_features

Baseline Features


['Forest_Area_ha',
 'Tree_Cover_Loss_ha',
 'Annual_Rainfall_mm',
 'Population_Density',
 'Fire_Incidents']

Feature engineering involves creating new variables from existing attributes to better represent relationships within the dataset.


In [7]:
# Creating Engineered Features

# Convert Illegal Logging into numeric values
df["Illegal_Logging_Flag"] = df["Illegal_Logging"].map({
    "Yes": 1,
    "No": 0
})

# Human pressure relative to forest size
df["Population_Pressure"] = (
    df["Population_Density"] /
    df["Forest_Area_ha"]
)

# Combined impact of fire and illegal logging
df["Fire_Logging_Impact"] = (
    df["Fire_Incidents"] *
    df["Illegal_Logging_Flag"]
)

print("New Features Created Successfully!")

display(
    df[
        [
            "Illegal_Logging_Flag",
            "Population_Pressure",
            "Fire_Logging_Impact"
        ]
    ].head()
)

New Features Created Successfully!


,Illegal_Logging_Flag,Population_Pressure,Fire_Logging_Impact
0,1.0,0.157197,16.0
1,1.0,0.128377,25.0
2,1.0,0.033032,16.0
3,1.0,0.082509,11.0
4,0.0,0.066062,0.0


In [8]:
# Saving the feature engineered dataset

output_file = "../data/engineered/deforestation_features.csv"

df.to_csv(output_file, index=False)

print(f"Dataset saved successfully to {output_file}")

Dataset saved successfully to ../data/engineered/deforestation_features.csv


In [10]:
# Load engineered dataset

df = pd.read_csv("../data/engineered/deforestation_features.csv")

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

categorical_columns = [
    "Region",
    "District",
    "Forest_Type",
    "Illegal_Logging",
    "Deforestation_Risk"
]

for col in categorical_columns:
    df[col] = encoder.fit_transform(df[col])

df.head()

,Record_ID,Observation_Date,Region,District,Forest_Type,Forest_Area_ha,Tree_Cover_Loss_ha,Annual_Rainfall_mm,Population_Density,Fire_Incidents,Illegal_Logging,Deforestation_Risk,Illegal_Logging_Flag,Population_Pressure,Fire_Logging_Impact
0,REC00001,2024-06-27,0,5,1,4179.46,235.27,1358.1,657.0,16,2,0,1.0,0.157197,16.0
1,REC00002,2021-04-30,1,2,1,2072.03,14.36,1605.3,266.0,25,2,0,1.0,0.128377,25.0
2,REC00003,2022-01-20,0,4,5,7417.11,543.23,1471.4,245.0,16,2,0,1.0,0.033032,16.0
3,REC00004,2021-07-17,3,3,2,5260.01,281.86,1609.9,434.0,11,2,0,1.0,0.082509,11.0
4,REC00005,2023-07-07,1,2,0,9793.89,575.77,1353.5,647.0,6,1,0,0.0,0.066062,0.0


Model evaluation

In [11]:
#training baseline model using original feateures
X = df[baseline_features]

y = df["Deforestation_Risk"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

baseline_model = RandomForestClassifier(
    random_state=RANDOM_STATE
)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_accuracy = accuracy_score(
    y_test,
    baseline_predictions
)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")

Baseline Accuracy : 0.8393


In [12]:
#training engineered model using engineered features and original features
engineered_features = baseline_features + [
    "Illegal_Logging_Flag",
    "Population_Pressure",
    "Fire_Logging_Impact"
]

X = df[engineered_features]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE
)

engineered_model = RandomForestClassifier(
    random_state=RANDOM_STATE
)

engineered_model.fit(X_train, y_train)

engineered_predictions = engineered_model.predict(X_test)

engineered_accuracy = accuracy_score(
    y_test,
    engineered_predictions
)

print(f"Engineered Accuracy : {engineered_accuracy:.4f}")

Engineered Accuracy : 1.0000


In [13]:
comparison = pd.DataFrame({
    "Feature Set": [
        "Baseline Features",
        "Engineered Features"
    ],
    "Accuracy": [
        baseline_accuracy,
        engineered_accuracy
    ]
})

comparison

,Feature Set,Accuracy
0,Baseline Features,0.839286
1,Engineered Features,1.000000


The Random Forest classifier was trained using both the baseline and engineered feature sets. Comparing the resulting accuracies provides an indication of whether feature engineering improved predictive performance.

Feature engineering enables the model to capture relationships that may not be immediately apparent in the original variables, potentially improving its ability to classify deforestation risk.